# Active learning vs kinematics, old vs new MPPI settings

| name | folder | dynamics | MPPI `temperature` / `wz_std` |
|---|---|---|---|
| `kin_new` | `rosbags_10_14_26_kinematics_baseline` | kinematics | 0.1 / 0.5 (since 10/09) |
| `kin_old` | `rosbags_10_14_26_kinematics_oldmppi` | kinematics | 0.3 / 0.3 |
| `nn_new` | `rosbags_10_09_26_al_nn` | NN, `train_from_scratch` | 0.1 / 0.5 |
| `nn_old` | `rosbags_10_14_26_nn_old_mppi` | NN, `train_from_scratch` | 0.3 / 0.3 |

Per-trial metrics come from `al_metrics.py` (cached in `~/AL_scripts/al_analysis_cache/`). To recompute (cuda_end python, ROS sourced):
```
PYTHONPATH=~/CubeRover_waypoint/src/nav2_stack ~/cuda_end/bin/python al_metrics.py \
  kin_new=~/AL_scripts/rosbags_10_14_26_kinematics_baseline:kinematics \
  kin_old=~/AL_scripts/rosbags_10_14_26_kinematics_oldmppi:kinematics \
  nn_new=~/AL_scripts/rosbags_10_09_26_al_nn:nn  nn_old=~/AL_scripts/rosbags_10_14_26_nn_old_mppi:nn
```
**Metrics:**
1. **1-step:** the model that drove the trial predicting 0.1 s of motion from the applied command (`cmd_vel_out`), vs mocap. For NN runs, trial *i* uses `mlp128_trial{i-1}.pt`.
2. **2.5 s:** `/optimal_trajectory_model` (MPPI's belief) end pose vs the real pose 2.5 s later.
3. **10 s:** rolling over every `/plan`: mean distance of the next 10 s of driving from that plan, and the end-point error (real position vs the plan point at the same driven distance).
4. **Distance:** driven length / first plan length, and / straight start-to-goal distance.
5. **Arcs and gears:** gear changes, share of replans with a reverse leg, and arc held (first plan a single arc, ≤ 0.25 m deviation while > 1 m from the goal, no reverse).

**Excluded:** invalid and low-battery trials (speed < 0.88 of command or < 13.5 V), and for NN runs the 5 kinematics bootstrap trials.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import al_report as R
NAMES = ['kin_new', 'kin_old', 'nn_new', 'nn_old']
runs = {n: R.load(n) for n in NAMES}
U = {n: R.usable(r) for n, r in runs.items()}
COL = {'kin_new': '#9ecae1', 'kin_old': '#3182bd', 'nn_new': '#fdae6b', 'nn_old': '#e6550d'}
for n, r in runs.items():
    bad = [(t['bag'], t.get('invalid') or 'low battery') for t in r['trials'] if t.get('invalid') or t.get('low_batt')]
    print(f"{n:8s} {len(r['trials'])} trials, {len(U[n])} used; excluded: {bad}")

In [ ]:
def compare(a, b):
    R.print_summary(R.summary_rows(U[a], U[b], a, b), a, b)
    for n, k, h, dev in R.arc_rows([U[a], U[b]], [a, b]):
        print(f'  arcs {n}: {h}/{k} held, mean max deviation {dev[0]:.2f} m')
    for n, k, h in R.outcome_rows([U[a], U[b]], [a, b]):
        print(f'  reached {n}: {h}/{k}')

def boxes(names, keys=None):
    keys = keys or [m for m in R.METRICS]
    nc = 4; nr = int(np.ceil(len(keys) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(4.2 * nc, 3.2 * nr), squeeze=False)
    for ax, (key, label, unit, better) in zip(axs.ravel(), keys):
        data = [[t.get(key) for t in U[n] if t.get(key) is not None and np.isfinite(t.get(key))] for n in names]
        bp = ax.boxplot(data, patch_artist=True, showfliers=False)
        for patch, n in zip(bp['boxes'], names):
            patch.set_facecolor(COL[n])
        for j, d in enumerate(data, 1):
            ax.plot(np.random.normal(j, 0.05, len(d)), d, '.', color='k', alpha=0.35, ms=3)
        ax.set_xticks(range(1, len(names) + 1)); ax.set_xticklabels(names, fontsize=8)
        ax.set_title(label + (f' [{unit}]' if unit else '') + (' (lower = better)' if better == 'low' else ''), fontsize=8)
    for ax in axs.ravel()[len(keys):]:
        ax.axis('off')
    fig.tight_layout()

## 1. Kinematics: new MPPI settings (`kin_new`) vs old (`kin_old`)
Same day (10/14), same surface and planner, same dynamics, so the difference is only `temperature`/`wz_std`.

In [ ]:
compare('kin_new', 'kin_old')
boxes(['kin_new', 'kin_old'])

## 2. Neural network (active learning): new MPPI settings (`nn_new`) vs old (`nn_old`)
`nn_new` ran on 10/09 with the old GP explorer (every 20th pose) and before the surface preparation, so differences beyond MPPI are possible.

In [ ]:
compare('nn_new', 'nn_old')
boxes(['nn_new', 'nn_old'])

## 3. Best NN vs best kinematics (both old MPPI settings): `nn_old` vs `kin_old`

In [ ]:
compare('nn_old', 'kin_old')
boxes(['nn_old', 'kin_old'])

## 4. Gear changes: a few bad trials, or everywhere?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.arange(0, 41, 2)
for n in NAMES:
    g = np.array([t['gears'] for t in U[n]])
    ax.hist(np.clip(g, 0, 40), bins=bins, histtype='step', lw=2, color=COL[n],
            label=f'{n}: {np.mean(g == 0)*100:.0f}% with none, {np.mean(g >= 5)*100:.0f}% with 5+')
ax.set_xlabel('gear changes per trial (clipped at 40)'); ax.set_ylabel('trials'); ax.legend(fontsize=8)

## 5. NN learning curves over the trials (rolling mean of 7 trials)

In [ ]:
keys = [('step1_yaw_rmse', '1-step yaw-rate error [rad/s]'), ('pred25_head_deg', '2.5 s heading error [deg]'),
        ('plan10_xtrack_m', '10 s distance from plan [m]'), ('gears', 'gear changes'), ('time', 'drive time [s]')]
fig, axs = plt.subplots(1, len(keys), figsize=(4 * len(keys), 3.2))
for ax, (k, lab) in zip(axs, keys):
    for n in ('nn_new', 'nn_old'):
        tr = sorted(U[n], key=lambda t: t['i'])
        i = np.array([t['i'] for t in tr]); v = np.array([t.get(k, np.nan) for t in tr], float)
        ax.plot(i, v, '.', color=COL[n], alpha=0.4)
        if len(v) >= 7:
            ax.plot(i[3:-3], np.convolve(v, np.ones(7) / 7, 'valid'), '-', color=COL[n], lw=2, label=n)
    for n in ('kin_old', 'kin_new'):
        ax.axhline(np.nanmean([t.get(k, np.nan) for t in U[n]]), color=COL[n], ls='--', lw=1, label=f'{n} mean')
    ax.set_title(lab, fontsize=9); ax.set_xlabel('trial')
axs[0].legend(fontsize=7)
fig.tight_layout()

## 6. The learned models themselves: each run's final model (`trial50`) on every folder's data (1-step)

In [ ]:
for n in ('nn_new', 'nn_old'):
    print(f"final model of {n} ({runs[n]['final_model']}):")
    for k, v in runs[n]['final_model_on'].items():
        print(f"   on {k:8s}: yaw-rate RMSE {v['yaw_rmse']:.4f} rad/s, speed RMSE {v['speed_rmse']:.4f} m/s  (n={v['n']})")
print('for reference, 1-step yaw-rate error of the model that drove each run:',
      {n: round(float(np.mean([t['step1_yaw_rmse'] for t in U[n] if 'step1_yaw_rmse' in t])), 3) for n in NAMES},
      '(kinematics runs: kinematics; NN runs: the checkpoint active in each trial)')

## Caveats
- **Each run chose its own goals** (GP explorer), but goal difficulty is similar: first plan 3.8–3.9 m, a single arc in 40–55% of trials.
- **`nn_new` is from 10/09** (old explorer, before the surface preparation); the other three are from 10/14.
- **The NN runs' first 5 trials are kinematics** and are excluded; kinematics runs use all trials.
- **The 2.5 s error measures MPPI's belief about its own chosen trajectory**, which starts from the measured speed at step 0.
- **Arc counts are small** (14–20 arc trials per run).